# 02 — PyTorch overview

PyTorch gives us three things: **tensors** (GPU-capable n-d arrays), **autograd**
(automatic gradients) and **`torch.nn`** (layers, losses, optimizers).

In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(torch.__version__, device)

2.14.0+cu130 cpu


## CPU vs GPU performance

A GPU has thousands of simple cores, so large element-wise operations and matrix
multiplies are massively faster there. For tiny tensors the launch overhead dominates
and the CPU can even win. (On a CPU-only machine both timings below use the CPU.)

In [2]:
def sync():
    if device == 'cuda':
        torch.cuda.synchronize()

a = torch.rand(100, 100, 100, 100, device=device)
b = torch.rand(100, 100, 100, 100, device=device)
start = time.time()
c = a * b
sync()
print(f'torch ({device}): {time.time() - start:.6f}s')

a_np = np.random.rand(100, 100, 100, 100)
b_np = np.random.rand(100, 100, 100, 100)
start = time.time()
c_np = a_np * b_np
print(f'numpy (cpu):   {time.time() - start:.6f}s')

torch (cpu): 0.075603s


numpy (cpu):   3.873931s


In [3]:
m1 = torch.rand(2048, 2048, device=device)
m2 = torch.rand(2048, 2048, device=device)
start = time.time()
for _ in range(10):
    m1 @ m2
sync()
print(f'10 matmuls 2048x2048 on {device}: {time.time() - start:.3f}s')

10 matmuls 2048x2048 on cpu: 0.333s


## More PyTorch functions we will use

In [4]:
print(torch.randint(-100, 100, (6,)))            # random integers
print(torch.tensor([[0.1, 1.2], [2.2, 3.1]]))    # from python data
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.empty(2, 3).shape)                   # uninitialised memory
print(torch.arange(5))
print(torch.linspace(3, 10, steps=5))
print(torch.logspace(-10, 10, steps=5))
print(torch.eye(3))
print(torch.empty_like(torch.zeros(2, 2)).shape)

tensor([ 59,  74,  48,  37,  -8, -66])
tensor([[0.1000, 1.2000],
        [2.2000, 3.1000]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
torch.Size([2, 3])
tensor([0, 1, 2, 3, 4])
tensor([ 3.0000,  4.7500,  6.5000,  8.2500, 10.0000])
tensor([1.0000e-10, 1.0000e-05, 1.0000e+00, 1.0000e+05, 1.0000e+10])
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
torch.Size([2, 2])


In [5]:
# multinomial: sample indices according to a probability distribution (used in generate)
probabilities = torch.tensor([0.1, 0.9])
samples = torch.multinomial(probabilities, num_samples=10, replacement=True)
print(samples)

# cat: concatenate along an existing dimension (append a sampled token)
tensor = torch.tensor([1, 2, 3, 4])
print(torch.cat((tensor, torch.tensor([5])), dim=0))

# tril / triu: lower / upper triangular — the causal mask of attention
print(torch.tril(torch.ones(5, 5)))
print(torch.triu(torch.ones(5, 5)))

# masked_fill: -inf where the mask is 0, then exp -> 0 after softmax
out = torch.zeros(5, 5).masked_fill(torch.tril(torch.ones(5, 5)) == 0, float('-inf'))
print(out)
print(torch.exp(out))

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
tensor([1, 2, 3, 4, 5])
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])


In [6]:
# transpose swaps two dims; stack adds a new dim
x = torch.zeros(2, 3, 4)
print(x.transpose(0, 2).shape)
print(torch.stack([torch.tensor([1, 2, 3])] * 3))

# nn.Linear: y = x @ W.T + b  (learnable)
sample = torch.tensor([10., 10., 10.])
linear = nn.Linear(3, 3, bias=False)
print(linear(sample))

# softmax: exponentiate then normalise so the outputs sum to 1
t = torch.tensor([1.0, 2.0, 3.0])
print(F.softmax(t, dim=0), F.softmax(t, dim=0).sum())

torch.Size([4, 3, 2])
tensor([[1, 2, 3],
        [1, 2, 3],
        [1, 2, 3]])
tensor([-5.0043,  1.6244,  0.8891], grad_fn=<SqueezeBackward4>)
tensor([0.0900, 0.2447, 0.6652]) tensor(1.)


## Embedding vectors

An **embedding** is a learnable lookup table of shape `(num_embeddings, embedding_dim)`.
Token id `i` → row `i`. Similar tokens end up with similar vectors during training.

In [7]:
vocab_size = 80
embedding_dim = 6
embedding = nn.Embedding(vocab_size, embedding_dim)

input_indices = torch.LongTensor([1, 5, 3, 2])
embedded_output = embedding(input_indices)
print(embedded_output.shape)
print(embedded_output)
# an embedding lookup is identical to one-hot @ weight matrix
one_hot = F.one_hot(input_indices, vocab_size).float()
print(torch.allclose(one_hot @ embedding.weight, embedded_output))

torch.Size([4, 6])
tensor([[-0.4105, -0.1300, -0.3756, -0.0659, -1.6323, -0.3982],
        [ 0.1850,  0.0385, -1.7774, -1.7728,  0.4769,  0.7326],
        [-0.1595,  0.2114, -0.8795,  0.4390, -0.1045, -1.4672],
        [ 0.5155, -1.6219, -0.6452,  0.5579, -0.5299, -0.1215]],
       grad_fn=<EmbeddingBackward0>)
True


## Dot product and matrix multiplication

Dot product: `a·b = Σ aᵢbᵢ` — a similarity score (used by attention: query·key).
Matrix multiply: entry `(i, j)` of `A @ B` is the dot product of row *i* of A with
column *j* of B, so shapes must agree: `(m×n) @ (n×p) → (m×p)`.

In [8]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])
print('dot:', torch.dot(a, b))           # 1*4 + 2*5 + 3*6 = 32

A = torch.tensor([[1, 2], [3, 4], [5, 6]])   # 3x2
B = torch.tensor([[7, 8, 9], [10, 11, 12]])  # 2x3
print(A @ B)                                  # 3x3
print(torch.matmul(A, B))                     # same thing

# batched matmul: leading dims are broadcast — this is how (B,T,C) @ (B,C,T) works
q = torch.rand(4, 8, 16)
k = torch.rand(4, 8, 16)
print((q @ k.transpose(-2, -1)).shape)

dot: tensor(32.)
tensor([[ 27,  30,  33],
        [ 61,  68,  75],
        [ 95, 106, 117]])
tensor([[ 27,  30,  33],
        [ 61,  68,  75],
        [ 95, 106, 117]])
torch.Size([4, 8, 8])


## Int vs float

Matmul requires both operands to have the same dtype. Token ids are `int64`, weights are
`float32` — mixing them raises an error. Cast with `.float()` / `dtype=`.

In [9]:
int_64 = torch.randint(1, (3, 2)).float()
float_32 = torch.rand(2, 3)
print(int_64.dtype, float_32.dtype)
print(int_64 @ float_32)

try:
    torch.randint(1, (3, 2)) @ torch.rand(2, 3)
except RuntimeError as e:
    print('error:', e)

torch.float32 torch.float32
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
error: expected m1 and m2 to have the same dtype, but got: long int != float


## Shapes: `view` / reshaping

`cross_entropy` expects `(N, C)` logits and `(N,)` targets, so we flatten `(B, T, C)`
into `(B*T, C)` with `.view`.

In [10]:
a = torch.rand(2, 3, 5)
B, T, C = a.shape
print(a.view(B * T, C).shape)

torch.Size([6, 5])
